In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("advancedReg.ipynb")

## Lecture Section

In this lecture, we will cover advanced topics in linear regression.
We will cover:
* Multiple linear regression
    * Adding predictors
* Categorical variables & dummy coding
    * Interaction terms
* Multicollinearity
* Evaluating multiple regression models
    * Adjusted R-squared, AIC, BIC
    * Comparing Nested Models
        * F-test
        * ANOVA

We are going to use the `penguins` dataset this time — it has a mix of continuous and categorical predictors.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson

penguins = sns.load_dataset("penguins").dropna()
penguins.head()

### Multiple Linear Regression

Last lecture we used **one** predictor to explain a response variable. In practice, most outcomes are influenced by **multiple** factors at once.

**Multiple linear regression** extends the simple model by adding more predictors:

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p + \varepsilon$$

Each $\beta_j$ represents the expected change in $y$ for a **one-unit increase in $x_j$, holding all other predictors constant**. That "holding constant" part is key — it's what makes multiple regression different from running separate simple regressions.

Let's predict `body_mass_g` from `flipper_length_mm` and `bill_length_mm`. We can compare to a simpler model that predicts with only `flipper_length_mm`.

In [ ]:
model_simple = smf.ols("body_mass_g ~ flipper_length_mm", data=penguins).fit()

print(model_simple.summary())

In [ ]:
model_multi  = smf.ols("body_mass_g ~ flipper_length_mm + bill_length_mm", data=penguins).fit()

print(model_multi.summary())

Notice how the coefficient for `flipper_length_mm` change when we add `bill_length_mm`. This is because the two predictors share some overlapping information — adding the second predictor adjusts the first one's estimate to account for that overlap.

Also notice the **Adjusted R-squared** — this is what we should use when comparing models with different numbers of predictors. Regular R-squared always increases when you add a variable, even a useless one; adjusted R-squared dulls this.

In [ ]:
print(f"Simple model  — R²: {model_simple.rsquared:.4f}, Adj. R²: {model_simple.rsquared_adj:.4f}")
print(f"Multiple model — R²: {model_multi.rsquared:.4f}, Adj. R²: {model_multi.rsquared_adj:.4f}")

### Categorical Variables & Dummy Coding

How do we handle categorical variables like `species`?

The answer is **dummy coding** (also called one-hot encoding). For a variable with $k$ categories, we create $k-1$ binary  columns. The omitted category becomes the **reference group** — all coefficients are interpreted relative to it.

`statsmodels` handles this automatically when you wrap a variable in `C()` in the formula. Let's add species to our model.

In [ ]:
model_cat = smf.ols("body_mass_g ~ flipper_length_mm + C(species)", data=penguins).fit()
print(model_cat.summary())

The reference species is **Adelie** (alphabetically first). The coefficients for `C(species)[T.Chinstrap]` and `C(species)[T.Gentoo]` tell us how much heavier or lighter those species are on average, **after controlling for flipper length**.

You can change the reference category by specifying it explicitly:

In [ ]:
# Make Gentoo the reference group
model_cat_ref = smf.ols("body_mass_g ~ flipper_length_mm + C(species, Treatment('Gentoo'))", data=penguins).fit()
print(model_cat_ref.params)

Let's visualize what including species does — we should now see three roughly parallel lines, one per species.

In [ ]:
plt.figure(figsize=(9, 5))
palette = {"Adelie": "steelblue", "Chinstrap": "coral", "Gentoo": "mediumseagreen"}

for species, grp in penguins.groupby("species"):
    plt.scatter(grp["flipper_length_mm"], grp["body_mass_g"],
                alpha=0.5, label=species, color=palette[species])
    x_range = np.linspace(grp["flipper_length_mm"].min(), grp["flipper_length_mm"].max(), 100)
    pred_df = pd.DataFrame({"flipper_length_mm": x_range, "species": species})
    plt.plot(x_range, model_cat.predict(pred_df), color=palette[species], linewidth=2)

plt.title("Body Mass vs. Flipper Length by Species")
plt.xlabel("Flipper Length (mm)")
plt.ylabel("Body Mass (g)")
plt.legend(title="Species")
plt.tight_layout()
plt.show()

### Interaction Terms

The parallel lines above assume that the **slope** of flipper length is the same for all species — only the intercept shifts. But what if the relationship between flipper length and body mass is steeper for some species than others?

We can test this with an **interaction term**, written as `x1:x2` or `x1*x2` in the formula:

- `x1:x2` — adds only the interaction
- `x1*x2` — adds both main effects **and** the interaction (shorthand for `x1 + x2 + x1:x2`)

An interaction between a continuous and a categorical variable allows the **slope to differ by group**.

In [ ]:
model_interact = smf.ols("body_mass_g ~ flipper_length_mm*C(species)", data=penguins).fit()
print(model_interact.summary())

In [ ]:
plt.figure(figsize=(9, 5))

for species, grp in penguins.groupby("species"):
    plt.scatter(grp["flipper_length_mm"], grp["body_mass_g"],
                alpha=0.5, label=species, color=palette[species])
    x_range = np.linspace(grp["flipper_length_mm"].min(), grp["flipper_length_mm"].max(), 100)
    pred_df = pd.DataFrame({"flipper_length_mm": x_range, "species": species})
    plt.plot(x_range, model_interact.predict(pred_df), color=palette[species], linewidth=2)

plt.title("Body Mass vs. Flipper Length (with Interaction)")
plt.xlabel("Flipper Length (mm)")
plt.ylabel("Body Mass (g)")
plt.legend(title="Species")
plt.tight_layout()
plt.show()

The lines are no longer parallel — each species now has its own slope. Whether this improvement is worth the extra complexity is something we can test formally, which we'll do in the model selection section.

### Multicollinearity

**Multicollinearity** occurs when two or more predictors are highly correlated with each other. When two predictors correlate together, they are giving your model the same information, so it's like you have two of the same variable. You will want to remove variables that repeat information.

We can check for multicollinearity by looking at a correlation heatmap. The heatmap below shows that `flipper_length_mm` and `bill_length_mm` correlate strongly with one another. Whether it is strong enough to remove one of those variables might be better tested with a nested model (covered shortly).

In [ ]:
X = penguins[["flipper_length_mm", "bill_length_mm", "bill_depth_mm"]]

plt.figure(figsize=(6, 4))
sns.heatmap(X.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5)
plt.title("Predictor Correlation Matrix")
plt.tight_layout()
plt.show()

### Model Selection

When you have multiple candidate models, how do you decide which one to use? There are a few approaches.

#### AIC and BIC Comparison

We can compare any set of models fitted on the same response variable using AIC or BIC — lower is better. Unlike R-squared, they automatically penalize for model complexity.

https://scienceinsights.org/aic-and-bic-in-regression-formulas-and-differences/

In [ ]:
models = {
    "Flipper only":          smf.ols("body_mass_g ~ flipper_length_mm", data=penguins).fit(),
    "Flipper + Bill length": smf.ols("body_mass_g ~ flipper_length_mm + bill_length_mm", data=penguins).fit(),
    "Flipper + Species":     smf.ols("body_mass_g ~ flipper_length_mm + C(species)", data=penguins).fit(),
    "Flipper * Species":     smf.ols("body_mass_g ~ flipper_length_mm * C(species)", data=penguins).fit(),
}

comparison = pd.DataFrame({
    "Model": models.keys(),
    "Adj. R²": [m.rsquared_adj for m in models.values()],
    "AIC":     [m.aic          for m in models.values()],
    "BIC":     [m.bic          for m in models.values()],
    "# Params":[m.df_model + 1 for m in models.values()],
}).round(2)

print(comparison.to_string(index=False))

#### Nested Model F-Test

When one model is a **restricted version** of another (i.e., a nested model), we can formally test whether the extra parameters are worth adding using an **F-test**.

For example: does adding the interaction term significantly improve the species model?

- **Null hypothesis**: the extra parameters are all zero (the simpler model is sufficient)
- **Alternative**: at least one extra parameter is non-zero

A low p-value means the larger model *might* be worth the extra complexity.

In [ ]:
model_no_interact  = models["Flipper + Species"]
model_with_interact = models["Flipper * Species"]

ftest = model_with_interact.compare_f_test(model_no_interact)
print(f"F-statistic: {ftest[0]:.4f}")
print(f"p-value:     {ftest[1]:.4f}")
print(f"df:          {ftest[2]}")

We can do similar with an ANOVA

In [ ]:
import pandas as pd
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm


anovaResults = anova_lm(model_no_interact, model_with_interact)
print(anovaResults)


If the p-value is small (< 0.05), the interaction terms significantly improve the model and we should keep them. If not, the simpler parallel-slopes model is preferred — extra complexity isn't justified by the data.

## Assignment Section

We will use the **WHO Life Expectancy dataset**, which contains health, economic, and demographic indicators for 193 countries from 2000–2015. The response variable is `Life_Expectancy`.

Key columns:

| Column | Description |
|---|---|
| `Life_Expectancy` | Life expectancy in years |
| `Schooling` | Average years of schooling |
| `BMI` | Average BMI of the population |
| `Adult_Mortality` | Adult mortality rate per 1000 people |
| `Status` | `Developed` or `Developing` |

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

url = "https://raw.githubusercontent.com/Sid-149/Life-Expectancy-Predictor-Comparative-Analysis/main/Notebooks/Life%20Expectancy%20Data.csv"
df = pd.read_csv(url)

df.columns = (
    df.columns.str.strip()
              .str.replace(' ', '_')
              .str.replace('/', '_')
)
df = df.rename(columns={'Life_expectancy': 'Life_Expectancy'})

df = df[['Life_Expectancy', 'Schooling', 'BMI', 'Adult_Mortality', 'Status']].dropna()

print(f"Rows: {len(df)}")
df.head()

**Question 1.**

Fit a simple multiple regression model predicting `Life_Expectancy` from `Schooling`. Then fit a multiple regression model predicting `Life_Expectancy` from `Schooling`, `BMI`, and `C(Status)`. Return:

- `model_simple`: the fitted OLS model (simple)
- `model_multiple`: the fitted OLS model (multiple)
- `simple_adj_r2`: the adjusted R-squared, rounded to 4 decimal places
- `adjusted_adj_r2`: the adjusted R-squared, rounded to 4 decimal places
- `ftest_pval`: compare the simple model to the multiple regression model with an f-test. Return the p-value. Make sure you have the restricted model on the correct side!
- `better_mod`: return 'simple' or 'multiple', answering which model is 'better' after running the ftest

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

model_simple = ...
simple_adj_r2 = ...

model_multiple = ...
multiple_adj_r2 = ...

ftest_pval = ...
better_mod = ...

In [ ]:
grader.check("q1")

**Question 2.**

Create a correlation heatmap of the predictor variables given in X below. Note that the *y* variable is included to help you.

Return:

- `multicollinearity`: `True`/`False` depending on if multicollinearity is present
- `fixed_model`: the model (fitted) with all predictors except the one you (possibly) choose to remove due to multicollinearity. If determining between two correlated predictor variables, you will want to remove the variable that correlates less with the *y* variable.



In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

X = df[["Schooling", "BMI", "Adult_Mortality", "Life_Expectancy"]]

...
multicollinearity = ...
fixed_model = ...

In [ ]:
grader.check("q2")

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()